# Exercice 2 : Transfer Learning — Chat vs Chien
## TP2 — Réseaux de Neurones Convolutifs

---

**Objectif :** Classifier des images de chats et chiens en utilisant le Transfer Learning.

**On va comparer 4 modèles pré-entraînés :**
1. **MobileNetV2** — léger, rapide
2. **VGG16** — classique, lourd
3. **ResNet50** — avec connexions résiduelles
4. **EfficientNetB0** — optimisé, performant

**Pourquoi Transfer Learning ?**
Entraîner un CNN from scratch nécessite beaucoup de données et de temps.
Le Transfer Learning réutilise un modèle déjà entraîné sur ImageNet (14 millions d'images).

---

## Étape 1 — Activer le GPU

**Runtime → Change runtime type → GPU**

In [ ]:
import tensorflow as tf
print(f"GPU disponible : {tf.config.list_physical_devices('GPU')}")

## Étape 2 — Imports

In [ ]:
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import (
    MobileNetV2, VGG16, ResNet50, EfficientNetB0
)
from sklearn.metrics import classification_report
import pandas as pd

print("Prêt !")

---
## Étape 3 — Charger le dataset Cats vs Dogs

**Dataset :**
- 23 262 images couleur
- 2 classes : Chat (0) / Chien (1)
- Tailles variées → on va toutes les redimensionner en 224×224

In [ ]:
# Charger le dataset
(train_ds, val_ds, test_ds), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True
)

print(f"Train : {info.splits['train'].num_examples * 0.8:.0f} images")
print(f"Validation : {info.splits['train'].num_examples * 0.1:.0f} images")
print(f"Test : {info.splits['train'].num_examples * 0.1:.0f} images")

In [ ]:
# Afficher quelques images
plt.figure(figsize=(12, 5))
for i, (image, label) in enumerate(train_ds.take(10)):
    plt.subplot(2, 5, i+1)
    plt.imshow(image)
    plt.title('Chien' if label.numpy() == 1 else 'Chat')
    plt.axis('off')
plt.tight_layout()
plt.show()

## Étape 4 — Prétraitement

On redimensionne toutes les images en 224×224 (la taille attendue par les modèles pré-entraînés) et on normalise les pixels entre 0 et 1.

In [ ]:
# Taille cible
IMG_SIZE = 224

# Fonction de prétraitement
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0  # normaliser entre 0 et 1
    return image, label

# Appliquer à chaque dataset
train_ds = train_ds.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

print("Données prétraitées !")

---
## Étape 5 — Data Augmentation

Pour éviter le surapprentissage, on modifie les images d'entraînement de manière aléatoire :
- **RandomFlip** : retourner l'image horizontalement
- **RandomRotation** : tourner légèrement
- **RandomZoom** : zoomer légèrement

In [ ]:
# Data Augmentation
data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

print("Data Augmentation créée !")

---
## Étape 6 — Créer une fonction pour construire un modèle

Au lieu de réécrire le code pour chaque modèle, on crée une **fonction** qui prend en entrée le nom du modèle et renvoie le modèle prêt à être entraîné.

**Principe du Transfer Learning :**
1. Charger le modèle pré-entraîné (sans la couche de classification)
2. **Geler** les couches convolutionnelles (on ne les modifie pas)
3. Ajouter nos propres couches pour la classification binaire

In [ ]:
# Dictionnaire des modèles disponibles
MODELS = {
    "MobileNetV2": MobileNetV2,
    "VGG16": VGG16,
    "ResNet50": ResNet50,
    "EfficientNetB0": EfficientNetB0
}

def create_model(model_class):
    """Construit un modèle avec Transfer Learning"""

    # 1. Charger le modèle de base pré-entraîné sur ImageNet
    base_model = model_class(
        weights="imagenet",        # poids pré-entraînés
        include_top=False,         # sans la couche de classification
        input_shape=(224, 224, 3)  # taille d'entrée
    )

    # 2. Geler le modèle de base (on n'entraîne pas ces couches)
    base_model.trainable = False

    # 3. Construire le modèle complet
    model = Sequential([
        data_augmentation,              # augmentation de données
        base_model,                     # modèle pré-entraîné
        layers.GlobalAveragePooling2D(),# réduire les features
        layers.Dropout(0.3),            # régularisation
        layers.Dense(1, activation="sigmoid")  # sortie binaire
    ])

    # 4. Compiler
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",   # classification binaire
        metrics=["accuracy"]
    )

    return model

print("Fonction créée !")

### Test : créer un modèle MobileNetV2 pour voir

In [ ]:
# Créer un modèle MobileNetV2 pour voir la structure
test_model = create_model(MobileNetV2)
test_model.summary()

---
## Étape 7 — Entraîner et évaluer chaque modèle

On utilise **EarlyStopping** pour arrêter l'entraînement si le modèle ne s'améliore plus.

Pour chaque modèle, on fait :
1. Créer le modèle
2. L'entraîner
3. Évaluer sur le jeu de test
4. Sauvegarder les résultats

In [ ]:
# Early Stopping : arrêter si val_loss ne s'améliore plus pendant 3 epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True  # garder le meilleur modèle
)

def train_model(name, model):
    """Entraîne un modèle et renvoie l'historique + l'accuracy"""
    print("=" * 50)
    print(f"Entraînement : {name}")
    print("=" * 50)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        callbacks=[early_stop],
        verbose=1
    )

    # Évaluation sur le jeu de test
    loss, acc = model.evaluate(test_ds)
    print(f"\nTest accuracy {name} : {acc:.2%}\n")

    return history, acc

print("Fonction d'entraînement prête !")

In [ ]:
# Entraîner les 4 modèles un par un
results = {}     # pour stocker les accuracy
histories = {}   # pour stocker les historiques

for name, architecture in MODELS.items():
    # Créer le modèle
    model = create_model(architecture)

    # Entraîner
    history, accuracy = train_model(name, model)

    # Sauvegarder
    histories[name] = history
    results[name] = accuracy

---
## Étape 8 — Comparaison des résultats

In [ ]:
# Tableau comparatif
df = pd.DataFrame({
    "Modèle": list(results.keys()),
    "Accuracy": [f"{v:.2%}" for v in results.values()]
})

print("\nTableau comparatif :")
print(df.to_string(index=False))

In [ ]:
# Graphique en barres
noms = list(results.keys())
accs = list(results.values())

plt.figure(figsize=(10, 6))
bars = plt.bar(noms, accs, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], width=0.5)
plt.ylim(0.8, 1.0)
plt.ylabel('Accuracy')
plt.title('Comparaison des 4 modèles (Transfer Learning)', fontsize=16)

# Ajouter les valeurs au-dessus des barres
for bar, a in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{a:.2%}', ha='center', fontsize=13, fontweight='bold')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

---
## Étape 9 — Courbes d'apprentissage

Pour chaque modèle, on affiche les courbes d'accuracy et de loss pendant l'entraînement.

In [ ]:
# Afficher les courbes pour chaque modèle
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, name in enumerate(histories.keys()):
    row = i // 2
    col = i % 2

    h = histories[name].history

    # Courbe d'accuracy
    axes[row, col].plot(h['accuracy'], label='Train', linewidth=2)
    axes[row, col].plot(h['val_accuracy'], label='Validation', linewidth=2)
    axes[row, col].set_title(f'{name} — Accuracy', fontsize=14)
    axes[row, col].set_xlabel('Epoch')
    axes[row, col].set_ylabel('Accuracy')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Courbes de Loss
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, name in enumerate(histories.keys()):
    row = i // 2
    col = i % 2

    h = histories[name].history

    axes[row, col].plot(h['loss'], label='Train', linewidth=2)
    axes[row, col].plot(h['val_loss'], label='Validation', linewidth=2)
    axes[row, col].set_title(f'{name} — Loss', fontsize=14)
    axes[row, col].set_xlabel('Epoch')
    axes[row, col].set_ylabel('Loss')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Étape 10 — Classification Report (détail)

Le **classification report** donne plus de détails que la simple accuracy :
- **Precision** : parmi les images prédites "chien", combien sont vraiment des chiens ?
- **Recall** : parmi les vrais chiens, combien ont été détectés ?
- **F1-score** : moyenne de precision et recall

In [ ]:
# Prendre le dernier modèle entraîné (EfficientNetB0)
# et faire un rapport de classification détaillé

predictions = []
labels_true = []

for images, labels in test_ds:
    pred = model.predict(images, verbose=0)
    pred = (pred > 0.5).astype(int)  # seuil à 0.5
    predictions.extend(pred.flatten())
    labels_true.extend(labels.numpy())

print("Classification Report — EfficientNetB0")
print("=" * 50)
print(classification_report(labels_true, predictions, target_names=['Chat', 'Chien']))

---
## Étape 11 — Synthèse

**Quel modèle choisir ?**

| Modèle | Params | Avantage | Inconvénient |
|--------|--------|----------|---------------|
| MobileNetV2 | ~3.4M | Rapide, léger | Moins précis |
| VGG16 | ~138M | Classique, bien compris | Très lourd |
| ResNet50 | ~25.6M | Bonne précision | Plus lent |
| EfficientNetB0 | ~5.3M | Meilleur compromis | Nouveau, moins connu |

In [ ]:
# Résumé final
print("=" * 50)
print("RÉSUMÉ FINAL")
print("=" * 50)
print(f"Dataset : Cats vs Dogs (23 262 images)")
print(f"Méthode : Transfer Learning")
print(f"Epochs : 10 (avec Early Stopping)")
print()

for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:20s} : {acc:.2%}")

print()
meilleur = max(results, key=results.get)
print(f"Meilleur modèle : {meilleur} ({results[meilleur]:.2%})")

---
## Résumé général

| Concept | Description |
|---------|-------------|
| **Transfer Learning** | Réutiliser un modèle pré-entraîné sur ImageNet |
| **Data Augmentation** | Augmenter la diversité du dataset (flip, rotation, zoom) |
| **Early Stopping** | Arrêter l'entraînement si le modèle ne s'améliore plus |
| **Binary Crossentropy** | Fonction de perte pour la classification binaire |
| **GlobalAveragePooling** | Réduire les features sans Flatten |

**Pourquoi le Transfer Learning est efficace ?**
- Les premières couches des CNN apprennent des features universelles (contours, textures)
- On réutilise ces features et on ne réentraîne que le classifieur
- Résultat : entraînement rapide et bonne précision